In [1]:
from astropy.table import Table
from astropy.constants import c
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.visualization.wcsaxes import SphericalCircle

from get_cutouts import get_cutout

import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

from tqdm import tqdm

from galaxy_selection import *

from velocity_map_fxns import *

## import data

In [2]:
loa = Table.read('/pscratch/sd/d/dbustos/rot_curves/shredded_VI.fits')
loa[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,unique_obs,Velocity,V_err,Z_center,VI,deproj_dist,deproj_r26
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,int64,float64,float64
6521555517440,1157225,204.22749,32.09493,0.010016011261740405,0.010016011261740405,0,71.96394567680545,0.002217102021260331,0.36627155666222244,65.12960815429688,114.35180431415512,49.22219615985824,1,0.010016039025980373,3.0,-2.9046128078129714,4248.5806387633265,0.010025824882245725,0,0.002881332554681444,0.4760043290496788
67815440646152,993764,131.54147036553024,19.348963933595506,0.6888593799981794,0.6888593799981794,4,7.544203393161297,0.004898285806031614,0.44172182227817786,9.1672945022583,188.95042393619124,179.78312943393294,1,0.688859381126886,3.0,199184.86718681874,206561.79926844206,0.014690024421240908,0,0.00677902926522777,0.6113251204383032
169678387281926,1430874,257.7495971,56.9324928,0.028921517883227336,0.028921517883227336,0,121857.89663615823,0.003250448915035935,0.31250071154567677,6.557606220245361,206.22810010342067,199.6704938831753,1,0.02892152786179073,3.0,158.1223507935872,12147.438163427121,0.028379120392506063,0,0.008096415535868652,0.7783957484224773
234526940856326,5001181,151.1772061023157,1.833743864948717,0.03108437229025544,0.03108437229025544,0,13100.219412088394,0.001016580293940661,0.34937980530094326,107.75557708740234,118.54239042958262,10.786813342180281,1,0.031084381613582526,4.0,20.42217921981953,20799.955384114714,0.06202829814864271,0,0.0031839814928551905,1.0942754258430336
234563162865665,632204,150.23137029728193,3.151162371010314,1.2707527411865653,1.2707527411865653,4,1.0103464424610138,0.011775468159487008,0.8630857807133053,159.4539031982422,322.60583622868717,163.15193303044498,1,1.270752742292688,23.0,361581.49656812777,381265.8408514983,0.05075534924045021,0,0.013838610970700105,1.014304330992709


In [3]:
SGA = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', 'ELLIPSE')
SGA[:5]

SGA_ID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,Z_LEDA,SB_D25_LEDA,MAG_LEDA,BYHAND,REF,GROUP_ID,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,BRICKNAME,RA,DEC,D26,D26_REF,PA,BA,RA_MOMENT,DEC_MOMENT,SMA_MOMENT,G_SMA50,R_SMA50,Z_SMA50,SMA_SB22,SMA_SB22.5,SMA_SB23,SMA_SB23.5,SMA_SB24,SMA_SB24.5,SMA_SB25,SMA_SB25.5,SMA_SB26,G_MAG_SB22,R_MAG_SB22,Z_MAG_SB22,G_MAG_SB22.5,R_MAG_SB22.5,Z_MAG_SB22.5,G_MAG_SB23,R_MAG_SB23,Z_MAG_SB23,G_MAG_SB23.5,R_MAG_SB23.5,Z_MAG_SB23.5,G_MAG_SB24,R_MAG_SB24,Z_MAG_SB24,G_MAG_SB24.5,R_MAG_SB24.5,Z_MAG_SB24.5,G_MAG_SB25,R_MAG_SB25,Z_MAG_SB25,G_MAG_SB25.5,R_MAG_SB25.5,Z_MAG_SB25.5,G_MAG_SB26,R_MAG_SB26,Z_MAG_SB26,SMA_SB22_ERR,SMA_SB22.5_ERR,SMA_SB23_ERR,SMA_SB23.5_ERR,SMA_SB24_ERR,SMA_SB24.5_ERR,SMA_SB25_ERR,SMA_SB25.5_ERR,SMA_SB26_ERR,G_MAG_SB22_ERR,R_MAG_SB22_ERR,Z_MAG_SB22_ERR,G_MAG_SB22.5_ERR,R_MAG_SB22.5_ERR,Z_MAG_SB22.5_ERR,G_MAG_SB23_ERR,R_MAG_SB23_ERR,Z_MAG_SB23_ERR,G_MAG_SB23.5_ERR,R_MAG_SB23.5_ERR,Z_MAG_SB23.5_ERR,G_MAG_SB24_ERR,R_MAG_SB24_ERR,Z_MAG_SB24_ERR,G_MAG_SB24.5_ERR,R_MAG_SB24.5_ERR,Z_MAG_SB24.5_ERR,G_MAG_SB25_ERR,R_MAG_SB25_ERR,Z_MAG_SB25_ERR,G_MAG_SB25.5_ERR,R_MAG_SB25.5_ERR,Z_MAG_SB25.5_ERR,G_MAG_SB26_ERR,R_MAG_SB26_ERR,Z_MAG_SB26_ERR,G_COG_PARAMS_MTOT,G_COG_PARAMS_M0,G_COG_PARAMS_ALPHA1,G_COG_PARAMS_ALPHA2,G_COG_PARAMS_CHI2,R_COG_PARAMS_MTOT,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT
int64,bytes16,bytes29,int64,float64,float64,bytes21,float32,float32,float32,float32,float32,float32,bool,bytes13,int64,bytes35,int16,bool,float64,float64,float32,bytes8,float64,float64,float32,bytes4,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32
2,SGA-2020 2,PGC1283207,1283207,228.3770865,5.4232017,S?,152.2,0.36307806,0.724436,0.03463229,23.40448,16.976,False,LEDA-20181114,0,PGC1283207,1,True,228.3770865,5.4232017,0.36307806,2283p055,228.3770803831908,5.423191398593787,0.49470574,SB26,158.20142,0.545691,228.37700918822188,5.4232652570544015,10.897086,3.3509698,3.1147978,3.240862,5.902337,6.9126143,7.941369,8.997992,10.073601,11.199986,12.391357,13.561038,14.841172,16.966799,16.108246,15.486356,16.879545,16.024958,15.400715,16.818878,15.967034,15.341793,16.776297,15.925804,15.300776,16.746685,15.897334,15.272053,16.725166,15.876816,15.2521105,16.708357,15.862035,15.237181,16.696539,15.851936,15.226998,16.689613,15.844313,15.21976,0.013392451,0.02354,0.021872982,0.01736985,0.024445537,0.039866067,0.05026544,0.08455789,0.122911856,0.005682776,0.0054258136,0.0049038026,0.005588406,0.005323561,0.0047632363,0.00543534,0.005177031,0.0046343105,0.0053025587,0.005040888,0.0045181247,0.005206092,0.0049438984,0.0044374703,0.0051483097,0.0048758644,0.0043834248,0.0051032505,0.0048264163,0.004344248,0.0050705094,0.004792021,0.004319857,0.005054293,0.004765629,0.0043044444,16.65942,0.34037337,0.2978292,3.0239506,0.07928849,15.820566,0.2640441,0.34559453,3.3033552,0.003811298,15.195567,0.29826432,0.3001073,3.2333765,0.011723555,0
3,SGA-2020 3,PGC1310416,1310416,202.54443750000002,6.9345944,Sc,159.26,0.4017908,0.7816278,0.073888786,23.498482,16.85,False,LEDA-20181114,1,PGC1310416,1,True,202.54443750000002,6.9345944,0.4017908,

In [4]:
SGA_dict = {}
for i in range(len(SGA)):
    SGA_dict[SGA['SGA_ID'][i]] = i

In [5]:
len(np.unique(loa['SGA_ID']))

4897

In [6]:
need_manual_VI = np.load('/pscratch/sd/d/dbustos/VI/ids_to_remove/manual_VI_ids.npy')
len(need_manual_VI)

191

In [7]:
loa['manual_VI'] = 0
for i in tqdm(need_manual_VI):
    loa['manual_VI'][loa['SGA_ID']==i] = 1

100%|██████████| 191/191 [00:00<00:00, 8119.76it/s]


## calculate rotational velocity

Quality Check </br>
1. ZWARN = 0 </br>
2. Delta Chi^2 > 25

If galaxies with center observation don't pass quality check, then also check if it has a symmetric observation

For symmetric observations
- Only use fibers that pass the quality check
- If no fibers pass quality check, then use all fibers
  
- If there are multiple pairs of observations, only use pairs with observations that fit the quality check
- If no pairs pass quality check, then use all pairs

In [8]:
# new error correcting for redrock 7km/s systematic uncertainty
dv_sys = 7 #km/s
dz_sys = dv_sys/c.to('km/s').value
loa['ZERR_MOD'] = np.sqrt(loa['Z_ERR']**2 + (dz_sys*(1 + loa['Z']))**2)

In [9]:
zwarn = 0
deltachi2 = 25

center_dist_lim = 0.001
unique_dist_lim = 0.01

q0 = 0.2

In [11]:
loa['Velocity'] = np.nan
loa['V_err'] = np.nan
loa['Z_center'] = np.nan

c = c.to('km/s')

for i in tqdm(np.unique(loa['SGA_ID'])):
    has_valid_center = False
    
    fiber = loa['SGA_ID'] == i 

    obs = loa[fiber]

    #find the index for this target in SGA
    sga_idx = SGA_dict[i]
    
    # find z and its error for all targets
    z_targ = loa['Z'][fiber]
    z_e = loa['ZERR_MOD'][fiber]

    # identify all points on semimajor axis
    on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]

    off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]

    #total_obs = obs['unique_obs'][0]

#-----------------------------------------------------------------------
# find center redshift if there is a center target that meets criteria
#-----------------------------------------------------------------------
    
    # check for center fiber
    if np.any(on_major['DIST_R26'] < center_dist_lim):
        
        # find all center fibers that fit quality check
        criteria_c = (on_major['DIST_R26'] < center_dist_lim)
        center = on_major[criteria_c & criteria_sym(on_major, zwarn, deltachi2)]

        # find z for center targets
        z_c = center['Z']

        # calculate weight of center targets
        zc_err = center['ZERR_MOD']
        weight = 1/(zc_err ** 2)
        #print(weight)
                
        # get weighted average
        # if weight = 0, then it did not meet criteria. 
        if np.any(weight > 0):
            has_valid_center = True
            z_cen = np.average(z_c, weights = weight)
            z_cen_err = np.sqrt(np.sum(zc_err ** 2))
    
#--------------------------------------------------------------
# find the center redshift given two symmetric points
#--------------------------------------------------------------
    if not has_valid_center:

        sga_id = SGA_dict[i]

        #split targets on semimajor axis onto either side of center galaxy
        if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

            left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

        else:
            left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
            
        left = on_major[left_index]
        right = on_major[~left_index]

        #----------------------------------------------------------
        # identify where the symmetric points are and group them ------------
        #----------------------------------------------------------
        #find R26 for each fiber
        right_dist, left_dist = np.array(right['DIST_R26']), np.array(left['DIST_R26'])

        #create a matrix subtracting each right r26 element from each left r26 element
        diff_matrix = np.abs(right_dist[:,np.newaxis]-left_dist)

        #identify all points in matrix where difference is within unique dist (fibers are symmetric)
        right_idx, left_idx = np.where(diff_matrix < unique_dist_lim)

        #make sure there are symmetric points
        if (len(right_idx) == 0) or (len(left_idx) == 0):  
            
            # if not, check if there are 10 unique points
            if len(obs) >= 10:

                distances = sorted(obs['deproj_r26'])
                counter = 0

                for i in range(len(distances)-1):
                    if distances[i+1] - distances[i] > unique_dist_lim:
                        counter += 1
            
                if counter >= 10:
                # get the average redshift
                    weight = 1/(z_e **2)
                    z_cen = np.average(z_targ, weights = weight)
                    z_cen_err = np.sqrt(np.sum(z_e**2))
                
                else:
                    print(sga_id, 'no center, symmetric point, or random pts')
                    continue  
        
        elif (len(right_idx) != 0) and (len(left_idx) != 0):
    
            #put those fibers into appropriate tables
            right['DIST_R26'] = np.round(right['DIST_R26'],6)
            left['DIST_R26'] = np.round(left['DIST_R26'],6)
            
            symmetric_right = right[np.unique(right_idx)].group_by('DIST_R26')
            symmetric_left = left[np.unique(left_idx)].group_by('DIST_R26')
    
            # print('left: ', symmetric_left['DIST_R26'], len(symmetric_left.groups))
            # print('right:', symmetric_right['DIST_R26'], len(symmetric_right.groups))
            
    #--------------------------------------
    # get pseudo-center for each grouping
    #--------------------------------------
            # number of fiber groups
            len_sym = len(symmetric_right.groups)
      
            #empty array for z_c and weight for each fiber group
            z_c = np.empty(len_sym)
            weight = np.empty(len_sym)
            
        #-----------------------   
        # for each fiber group
        #-----------------------
            for z in range(len_sym):
                
                #-----------------------------------------
                # get right points
                #-----------------------------------------
                right_group = symmetric_right.groups[z]
                
                # check if there is more than one observation
                if len(right_group) > 1:
                    # if there is, only take the good observations
                    if np.any(right_group['ZWARN'] == 0) and np.any(right_group['DELTACHI2'] > 25):
                        right_group = right_group[criteria_sym(right_group, zwarn, deltachi2)]
                
                #-----------------------------------------
                # get left points
                #-----------------------------------------
                left_group = symmetric_left.groups[z]
                
                # check if there is more than one observation
                if len(left_group) > 1:
                    # if there is, only take the good observations
                    if np.any(left_group['ZWARN'] == 0) and np.any(left_group['DELTACHI2'] > 25):
                        left_group = left_group[criteria_sym(left_group, zwarn, deltachi2)]
                        
                #--------------------------------
                # get redshift and pseudo center
                #--------------------------------
                # get average z for right and left
                z_right = right_group['Z'].mean()
                z_left = left_group['Z'].mean()
    
                # pseudo z_center for fiber group
                z_c[z] = (z_right+z_left)/2
    
                # propagate uncertainty
                zr_err = np.sum(right_group['ZERR_MOD'] ** 2)
                zl_err = np.sum(left_group['ZERR_MOD'] ** 2)
                
                z_err = np.sqrt(zr_err + zl_err)/2
              
                # weight for each fiber group
                weight[z] = 1/(z_err ** 2)
    
        #----------------------------------------------------
        # if multiple symmetric groups, remove the bad pairs
        #----------------------------------------------------
            if len_sym > 1:
                
                # copy array of z_c and weight
                zc_pairs = z_c.copy()
                weight_pairs = weight.copy()
                
                for pair in range(len_sym):
                    right_group = symmetric_right.groups[pair]
                    right_group = right_group[criteria_sym(right_group, zwarn, deltachi2)]
                
                    left_group = symmetric_left.groups[pair]
                    left_group = left_group[criteria_sym(left_group, zwarn, deltachi2)]
    
                    # if all fibers in the group are bad, then remove it from redshift and weight calculation
                    
                    if (len(right_group) == 0) or (len(left_group) == 0):
                        zc_pairs[pair] = 0
                        weight_pairs[pair] = 0
                        
                # if every pair is bad, then keep all the pairs
                if np.sum(zc_pairs) == 0:
                    print(i,'all bad pairs zc')
                    z_c = z_c
                else:
                    z_c = zc_pairs
    
                if np.sum(weight_pairs) == 0:
                    print(i,'all bad pair weight')
                    weight = weight
                else:
                    weight = weight_pairs 
    
        #-----------------------------------------------------
        # calculate weighted pseudo z_center and error
        #-----------------------------------------------------
            z_cen = np.average(z_c, weights = weight)
            
            z_cen_err = np.sqrt(np.sum(z_err**2))
    
            # print('pseudo:',z_cen)
    
#---------------------------------------------------------
#find the redshift of each fiber relative to the center
#---------------------------------------------------------

    # relative redshift
    z_rel = (1 + z_targ)/(1 + z_cen) - 1

    # inclination angle
    axis_ratio = SGA['BA'][sga_idx]
    inc = sin_i(axis_ratio, q0)

    #angle off axis
    phi = loa['ANGLE_OFF_AXIS'][fiber]
    cosphi = np.cos(np.radians(phi))
      
    # find the rotational velocity
    # adjusted for inclination and angle off major axis
    velocity = z_rel*c/(inc * cosphi)

    # rotational velocity error
    v_error = c*np.sqrt((z_cen_err**2)+(z_e**2))/(inc * cosphi)

    counter = np.where(abs(velocity.value) < 1000, 1, 0)

    
    # make sure all galaxies have 3 points with velocities < 1000 km/s
    # unless its marked that I still need to VI later
    
    manual_VI = obs['manual_VI'][0]
    
    if (np.sum(counter) < 3 & manual_VI ==0):
        #print (sga_idx, ':not enough points')
        loa['Velocity'][fiber] = np.nan
        loa['V_err'][fiber] = np.nan

    else:
        loa['Velocity'][fiber] = velocity  

        loa['V_err'][fiber] = v_error
    
        loa['Z_center'][fiber] = z_cen_err
    #print(velocity)

  6%|▌         | 279/4897 [00:00<00:14, 311.00it/s]

57891 all bad pairs zc
57891 all bad pair weight


 10%|█         | 495/4897 [00:01<00:12, 345.34it/s]

32756 no center, symmetric point, or random pts


 12%|█▏        | 603/4897 [00:01<00:12, 349.64it/s]/global/common/software/nersc/pe/conda-envs/26.1.0/python-3.13/nersc-python/lib/python3.13/site-packages/astropy/units/quantity.py:648: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
/global/common/software/nersc/pe/conda-envs/26.1.0/python-3.13/nersc-python/lib/python3.13/site-packages/astropy/units/quantity.py:648: RuntimeWarning: invalid value encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
 21%|██        | 1004/4897 [00:03<00:14, 269.93it/s]

...

 57%|█████▋    | 2802/4897 [00:07<00:04, 423.09it/s]

819390 all bad pairs zc
819390 all bad pair weight


 68%|██████▊   | 3350/4897 [00:09<00:03, 420.76it/s]

958312 all bad pairs zc
958312 all bad pair weight


100%|██████████| 4897/4897 [00:13<00:00, 360.85it/s]


## redo criteria cut

In [12]:
# create a new table that removes all NAN
loa_new = loa[~np.isnan(loa['Velocity'])]
print(len(np.unique(loa_new['SGA_ID'])))

4893


In [14]:
# redo criteria cut
#empty column to classify galaxy
loa_new['Selection'] = 0
# test_counter = 0

#for each unique SGA ID
for i in tqdm(np.unique(loa_new['SGA_ID'])):

    #identify all the galaxies and targets
    targ_id = loa_new['SGA_ID'] == i
    
    #makes a table of the targets corresponding to this galaxy
    targ = loa_new[targ_id]
    #print(targ)

    manual_VI = targ['manual_VI'][0]

    if manual_VI == 1:
        continue
    
    obs = targ[abs(targ['Velocity'] < 1000)]
    #print(obs)

    sga_id = SGA_dict[i] 

    criteria_a = False
    criteria_b = False


#if the galaxy has more than three observations
    if len(obs) >= 3:

#-----
# a
#-----
        #check to see if there is a center observation
        if np.any(obs['deproj_r26'] <= center_dist_lim):
            # test_counter += 1
        
            # check how many unique points on semimajor axis
            not_center = obs[obs['deproj_r26'] > unique_dist_lim]

            if len(not_center) >= 2 and (np.max(not_center['deproj_r26']) - np.min(not_center['deproj_r26'])) >= unique_dist_lim:
                
                #since there 2 unique points, classify it as viable galaxy
                loa_new['Selection'][targ_id] = 1
                criteria_a = True

# #----
# # b
# #----
        elif len(obs) >= 10:
            distances = sorted(obs['deproj_r26'])
            counter = 0

            for i in range(len(distances)-1):
                if distances[i+1] - distances[i] > unique_dist_lim:
                    counter += 1
            
            if counter >= 10:
                loa_new['Selection'][targ_id] = 2
                criteria_b = True

        
#----
# c
#----
          
        elif not (criteria_a or criteria_b):
            
            # identify all points on semimajor axis
            on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]

            # identify all points off major axis
            off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]


            if len(on_major) > 0:
            
                #split targets on semimajor axis onto either side of center galaxy
                if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

                    left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

                else:
                    left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
                
                left = on_major[left_index]
                right = on_major[~left_index]

            if len(left) > 0 and len(right) > 0:

                for j in range(len(left)):
                    
                    # check that there are 2 symmetric observations
                    if np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) < unique_dist_lim):

                    #check if there is a third point
                        if (np.any(np.abs(right['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim) 
                            or np.any(np.abs(left['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)
                            or np.any(np.abs(off_major['deproj_r26'] - left['deproj_r26'][j]) > unique_dist_lim)):

                            #viable galaxy
                            loa_new['Selection'][targ_id] = 3

100%|██████████| 4893/4893 [00:07<00:00, 654.04it/s]


In [15]:
loa_new = loa_new[loa_new['Selection']!=0]
print(len(np.unique(loa_new['SGA_ID'])))

4084


## velocity map

In [17]:
loa_new.write('/pscratch/sd/d/dbustos/rot_curves/loa_rot_velocity.fits',format='fits',overwrite=True)

In [ ]:
for sga_id in tqdm(np.unique(loa_new['SGA_ID'])):
    
    #identify all targets in the galaxy
    targ_list = loa_new[loa_new['SGA_ID']==sga_id]

    #find the index for this target in SGA
    sga_idx = SGA_dict[sga_id]
    
    #get coordinates for center of galaxy
    center_ra, center_dec = float(SGA['RA'][sga_idx]), float(SGA['DEC'][sga_idx])

    #get velocities for each point
    velocity = np.array(targ_list['Velocity'])

    #find max velocity for colormap
    v_max = max((v for v in abs(velocity) if v<=1000), default=1000)

#-------------------------------------------
# get cutout
#-------------------------------------------
    # D26 in arcmin
    d26 = SGA['D26'][sga_idx]
    
    pix = int(2 * d26*60/0.262)

    if (pix < 2500):
        npix = np.minimum(pix,1024)

    elif (pix > 2500):
        npix = np.minimum(pix,3000)

    img_file, wcs = get_cutout(sga_id, center_ra, center_dec, dir='/pscratch/sd/d/dbustos/loa_cutouts/cutouts/',dr=10,size=npix)
    img = mpl.image.imread(img_file)

    fig1 = plt.figure(figsize=(7,5))


    ax = fig1.add_subplot(111, projection=wcs)
    
    #add the cutout to make ellipse right size, but set opacity to 0
    ax.imshow(np.flip(img, axis=0),alpha=0)
    ax.coords[0].set_format_unit(u.deg)
    ax.set(xlabel='ra', ylabel='dec')
#---------------------------------
# generate ellipse
#---------------------------------
    
    #PA for ellipse in rad
    PA = np.radians(SGA['PA'][sga_idx])
    
    #radius semimajor axis in arcmin
    smajor = (d26/2)/60
    #radius semiminor axis in arcmin
    sminor = ((SGA['BA'][sga_idx]) * smajor)

    theta = np.linspace(0,2*np.pi,365)
    # ellipse coordinates before transformation
    smajor1 = smajor * np.cos(theta)
    sminor1 = sminor * np.sin(theta)

    # rotation matrix
    rot_ra = (smajor1*np.sin(PA) + sminor1*np.cos(PA))/np.cos(np.radians(center_dec))
    rot_dec = smajor1*np.cos(PA) - sminor1*np.sin(PA)
 
    #ellipse coordinates after transformation
    ellipse_ra = center_ra + rot_ra
    ellipse_dec = center_dec + rot_dec

    ax.plot(ellipse_ra,ellipse_dec,color = 'black',lw=2,transform= ax.get_transform('icrs'))

#----------------------------
    pa_minor = PA + np.pi/2

    minor_line = np.array([-sminor, sminor])

    # rotation matrix
    rot_line_ra = minor_line * np.sin(pa_minor) / np.cos(np.radians(center_dec))
    rot_line_dec = minor_line * np.cos(pa_minor)

    minor_line_ra = center_ra + rot_line_ra 

    minor_line_dec = center_dec + rot_line_dec 

    ax.plot(minor_line_ra, minor_line_dec,color = 'black',lw=2,ls = ':',alpha = .5,transform= ax.get_transform('icrs'))
#-------------------------------------------
# get ra and dec
#-------------------------------------------
    # velocities < 1000 km/s
    ra = np.array(targ_list['TARGET_RA'][velocity<1000]) 
    dec = np.array(targ_list['TARGET_DEC'][velocity<1000]) 

    # velocities < 1000 km/s
    ra_null = np.array(targ_list['TARGET_RA'][velocity>1000]) 
    dec_null = np.array(targ_list['TARGET_DEC'][velocity>1000]) 

#-------------------------------------------
# plot ra and dec
#-------------------------------------------

    ax1 = ax.scatter(np.array(ra), np.array(dec),
                     vmax = v_max, vmin= -v_max,
                     c = velocity[velocity < 1000], cmap = 'bwr', edgecolor = 'black',transform= ax.get_transform('icrs'))

    ax2 = ax.scatter(np.array(ra_null), np.array(dec_null),
                     vmax = v_max, vmin= -v_max,
                     c = velocity[velocity > 1000], cmap = 'bwr', marker = 'x',transform= ax.get_transform('icrs'))
    
    fig1.colorbar(ax1, ax=ax)
    
#-------------------------
# plot indices to fibers
#-------------------------
    loc_idx = {}
    
    for i, j in enumerate(targ_list):
        ra, dec = j['TARGET_RA'], j['TARGET_DEC']

        if (ra, dec) not in loc_idx:
            loc_idx[(ra,dec)] = []

        loc_idx[(ra,dec)].append(str(i))

    for i, j in loc_idx.items():
        combined = ', '.join(loc_idx[i])
        ax.text(i[0], i[1], combined, transform=ax.get_transform('icrs'), color='yellow', fontsize=7, fontweight = 'bold'
               ,bbox = dict(facecolor='black', alpha=0.5, edgecolor='white', lw=0.01, pad = .5))

    
    # note the galaxy still needs manual VI
    if targ_list['manual_VI'][0] == 1:
        ax.annotate('needs VI again', xy = (10,10), xycoords = 'figure pixels')
    
    ax.set_title('SGA ID: {}'.format(sga_id))
    fig1.subplots_adjust(top=0.85, right=0.85, bottom=0.15, left=0.15)
    
    fig1.savefig('/pscratch/sd/d/dbustos/velocity_maps/' + 'sga_{}_velocity_map.png'.format(sga_id), dpi=120)
    
    fig1.clear()
    plt.close(fig1)

 11%|█         | 456/4084 [01:31<12:05,  5.00it/s]

In [92]:
loa_new[loa_new['SGA_ID']==25835]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2,DIST,DIST_R26,PA,C_TO_F_ANGLE,ANGLE_OFF_AXIS,Selection,ZERR_MOD,Velocity,V_err,Z_center,VI,deproj_dist,deproj_r26,manual_VI
int64,int64,float64,float64,float64,float64,int64,float64,float64,float64,float64,float64,float64,int64,float64,float64,float64,float64,int64,float64,float64,int64
1070149704613890,25835,149.03188275971598,1.4781137750417956,0.36422285431599233,0.36422285431599233,4,5.430651303380728,0.0025011364303141725,0.6700000088378798,106.63904571533203,286.63904677175344,0.0,1,0.3642228557089186,122443.80773059952,135918.91326687956,0.04391417994108791,0,0.0025011364303141725,0.6700000088378798,0
1070149704613892,25835,149.035460678035,1.4770448523060304,0.025552152796404576,0.025552152796404576,0,43.45357794011943,0.0012319030179347937,0.3300000043580373,106.63904571533203,106.63904677148727,0.0,1,0.025552164016914354,71.65207769077189,18823.645510802933,0.04391417994108791,0,0.0012319030179347937,0.3300000043580373,0
1070149704613893,25835,149.0366771694792,1.4766814172641558,0.2932586074011604,0.2932586074011604,4,0.2291641600895673,0.0025011364303401122,0.6700000088448285,106.63904571533203,106.63904677156567,0.0,1,0.2932586089558535,96802.23959291095,109861.36299839756,0.04391417994108791,0,0.0025011364303401122,0.6700000088448285,0
1083343844147201,25835,149.03070204591484,1.4784665182804544,0.16866252099677648,0.16866252099677648,0,18.201470465399325,0.0037330394482635054,1.000000013199812,106.63904571533203,286.63904677159996,0.0,1,0.16866252320419317,51781.835965410646,64571.441543436966,0.04391417994108791,0,0.0037330394482635054,1.000000013199812,0
1083343844147206,25835,149.03785788138282,1.4763286708519079,0.4628997391055357,0.4628997391055357,4,4.170857340097427,0.0037330394482887786,1.0000000132065818,106.63904571533203,106.63904677147758,0.0,1,0.4628997403658152,158098.8054928187,172270.62936922544,0.04391417994108791,0,0.0037330394482887786,1.0000000132065818,0
39089837424255043,25835,149.0342799653704,1.4773975974455036,0.025353586446937228,0.025353586446937228,0,54241.568600058556,0.0,0.0,106.63904571533203,0.0,0.0,1,0.02535359775094609,-0.09606029531880644,18786.75438067468,0.04391417994108791,0,0.0,0.0,0
39089837487249741,25835,149.0342799653704,1.4773975974455036,0.02535424749308606,0.02535424749308606,0,74374.63736784458,0.0,0.0,106.63904571533203,0.0,0.0,1,0.025354258796814775,0.14279603883605685,18786.876836988155,0.04391417994108791,0,0.0,0.0,0
39627823467463477,25835,149.0342799653704,1.4773975974455036,0.02535372297417083,0.02535372297417083,0,72887.40761983395,0.0,0.0,106.63904571533203,0.0,0.0,1,0.025353734278121827,-0.04672880065951236,18786.77967163514,0.04391417994108791,0,0.0,0.0,0
